# Cross-Country Climate Comparison (2015–2026)

**Analysis Scope:** Ethiopia, Kenya, Sudan, Tanzania, Nigeria

**Objective:** Produce COP32-grade comparative analysis identifying regional vulnerability patterns, winners/losers from climate change, and coalition-building opportunities for Ethiopia to strengthen Africa's climate negotiating position.

**Framework:**
1. What is changing? (comparative trends)
2. What did it cause? (differential impacts)
3. What does it demand? (regional policy alignment)

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

DATA_DIR = 'data'
countries = ['Ethiopia', 'Kenya', 'Sudan', 'Tanzania', 'Nigeria']
print(f"Loading cleaned data for: {', '.join(countries)}")

In [ ]:
# Load all cleaned country datasets
dfs = []
for country in countries:
    path = os.path.join(DATA_DIR, f"{country.lower()}_clean.csv")
    try:
        df_country = pd.read_csv(path)
        df_country['Country'] = country
        dfs.append(df_country)
        print(f"✓ Loaded {country}: {len(df_country)} rows")
    except FileNotFoundError:
        print(f"⚠️  Missing: {path}")

# Concatenate all data
df_all = pd.concat(dfs, ignore_index=True)
df_all['DATE'] = pd.to_datetime(df_all['DATE'])
df_all['YearMonth'] = df_all['DATE'].dt.to_period('M')

print(f"\nCombined dataset: {df_all.shape}")
print(f"Date range: {df_all['DATE'].min()} to {df_all['DATE'].max()}")

## 1. Temperature Trend Comparison

**WHAT IS CHANGING?** Compare warming trends across countries.

In [ ]:
# Monthly temperature aggregation by country
temp_by_month = df_all.groupby(['Country', 'YearMonth'])['T2M'].mean().reset_index()
temp_by_month['YearMonth'] = temp_by_month['YearMonth'].dt.to_timestamp()

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
for country in countries:
    data = temp_by_month[temp_by_month['Country'] == country]
    ax.plot(data['YearMonth'], data['T2M'].rolling(12).mean(), 
           linewidth=2.5, marker='o', markersize=3, label=country, alpha=0.8)

ax.set_title('Temperature Trends — 5 African Countries (12-month moving average)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Average Temperature (°C)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature summary statistics by country
temp_summary = df_all.groupby('Country')['T2M'].agg([
    ('Mean (°C)', 'mean'),
    ('Median (°C)', 'median'),
    ('Std Dev', 'std'),
    ('Min (°C)', 'min'),
    ('Max (°C)', 'max')
]).round(2).sort_values('Mean (°C)', ascending=False)

print("\n=== TEMPERATURE SUMMARY BY COUNTRY ===")
print(temp_summary)

# Rank by variability (higher std = more unpredictable)
print("\n⚠️  Temperature Variability Ranking (higher = more unstable):")
variability_rank = df_all.groupby('Country')['T2M'].std().sort_values(ascending=False)
for i, (country, std_val) in enumerate(variability_rank.items(), 1):
    print(f"{i}. {country}: σ = {std_val:.2f}°C")

## 2. Precipitation Variability Comparison

**WHAT IS CHANGING?** Compare rainfall patterns and drought frequency.

In [ ]:
# Monthly precipitation aggregation
precip_by_month = df_all.groupby(['Country', 'YearMonth'])['PRECTOTCORR'].sum().reset_index()
precip_by_month['YearMonth'] = precip_by_month['YearMonth'].dt.to_timestamp()

# Boxplot comparison
fig, ax = plt.subplots(figsize=(10, 6))
df_all.boxplot(column='PRECTOTCORR', by='Country', ax=ax)
ax.set_title('Precipitation Variability by Country', fontsize=13, fontweight='bold')
ax.set_xlabel('Country', fontsize=11)
ax.set_ylabel('Daily Precipitation (mm)', fontsize=11)
plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

In [ ]:
# Precipitation summary
precip_summary = df_all.groupby('Country')['PRECTOTCORR'].agg([
    ('Mean Daily (mm)', 'mean'),
    ('Std Dev', 'std'),
    ('Coefficient of Variation', lambda x: x.std() / x.mean() if x.mean() != 0 else 0),
    ('Annual Total (mm)', lambda x: x.sum() * 365 / len(x) if len(x) > 0 else 0)
]).round(2).sort_values('Coefficient of Variation', ascending=False)

print("\n=== PRECIPITATION SUMMARY (Higher CV = More Unpredictable) ===")
print(precip_summary)
print("\n(CV: coefficient of variation = std / mean. Values >1 indicate highly skewed distributions)")

## 3. Extreme Event Frequency

**WHAT DOES IT DEMAND?** Quantify extreme events to justify policy interventions.

In [ ]:
# Count extreme heat days (T2M_MAX > 35°C)
extreme_heat = df_all[df_all['T2M_MAX'] > 35].groupby('Country').size().rename('Heat_Days_>35C')

# Count heavy rain days (PRECTOTCORR ≥ 50 mm)
heavy_rain = df_all[df_all['PRECTOTCORR'] >= 50].groupby('Country').size().rename('Heavy_Rain_Days_≥50mm')

# Count dry days (PRECTOTCORR < 1 mm)
dry_days = df_all[df_all['PRECTOTCORR'] < 1].groupby('Country').size().rename('Dry_Days_<1mm')

# Combine
extremes = pd.concat([extreme_heat, heavy_rain, dry_days], axis=1).fillna(0).astype(int)
extremes['Dry_Days_%'] = (extremes['Dry_Days_<1mm'] / df_all.groupby('Country').size() * 100).round(1)
extremes['Heat_Days_%'] = (extremes['Heat_Days_>35C'] / df_all.groupby('Country').size() * 100).round(1)

print("\n=== EXTREME EVENTS FREQUENCY ===")
print(extremes.sort_values('Heat_Days_%', ascending=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

extremes['Heat_Days_>35C'].sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='red', alpha=0.7)
axes[0].set_title('Extreme Heat Days (T2M_MAX > 35°C)', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Country')
axes[0].grid(True, alpha=0.3, axis='y')

extremes['Dry_Days_%'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='brown', alpha=0.7)
axes[1].set_title('Dry Days (% of total days)', fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Country')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Statistical Comparison

Test if temperature and precipitation differences are statistically significant.

In [ ]:
# One-way ANOVA for temperature
groups_temp = [df_all[df_all['Country'] == c]['T2M'].dropna().values for c in countries]
f_stat_temp, p_val_temp = stats.f_oneway(*groups_temp)

# One-way ANOVA for precipitation
groups_precip = [df_all[df_all['Country'] == c]['PRECTOTCORR'].dropna().values for c in countries]
f_stat_precip, p_val_precip = stats.f_oneway(*groups_precip)

print("\n=== STATISTICAL TESTING (One-way ANOVA) ===")
print(f"Temperature across countries:")
print(f"  F-statistic: {f_stat_temp:.4f}")
print(f"  P-value: {p_val_temp:.4e}")
print(f"  Significant difference: {'YES ✓' if p_val_temp < 0.05 else 'NO'}")

print(f"\nPrecipitation across countries:")
print(f"  F-statistic: {f_stat_precip:.4f}")
print(f"  P-value: {p_val_precip:.4e}")
print(f"  Significant difference: {'YES ✓' if p_val_precip < 0.05 else 'NO'}")

print("\nInterpretation: Small p-values (<0.05) indicate countries have significantly different climate patterns.")

## 5. Vulnerability Ranking & COP32 Findings

Create a composite vulnerability score to rank countries by climate stress.

In [ ]:
# Create vulnerability indicators
vulnerability = pd.DataFrame(index=countries)

# 1. Temperature trend (higher = warmer = higher stress)
temp_rank = df_all.groupby('Country')['T2M'].mean().rank(ascending=False)
vulnerability['Temp_Rank'] = temp_rank

# 2. Precipitation instability (higher CV = more drought risk)
precip_cv = df_all.groupby('Country')['PRECTOTCORR'].apply(
    lambda x: x.std() / x.mean() if x.mean() > 0 else 0
)
precip_cv_rank = precip_cv.rank(ascending=False)
vulnerability['Precip_Variability_Rank'] = precip_cv_rank

# 3. Extreme heat frequency (higher = more heat stress)
heat_rank = extremes['Heat_Days_%'].rank(ascending=False)
vulnerability['Heat_Stress_Rank'] = heat_rank

# 4. Drought frequency (higher = more dry days)
drought_rank = extremes['Dry_Days_%'].rank(ascending=False)
vulnerability['Drought_Rank'] = drought_rank

# Composite score (mean rank)
vulnerability['Composite_Score'] = vulnerability[[
    'Temp_Rank', 'Precip_Variability_Rank', 'Heat_Stress_Rank', 'Drought_Rank'
]].mean(axis=1).rank(ascending=False)

print("\n=== CLIMATE VULNERABILITY RANKING ===")
print("(Rank 1 = Most vulnerable, Rank 5 = Least vulnerable)")
print(vulnerability.sort_values('Composite_Score'))

print("\n" + "="*60)
most_vulnerable = vulnerability['Composite_Score'].idxmin()
least_vulnerable = vulnerability['Composite_Score'].idxmax()
print(f"🔴 MOST VULNERABLE: {most_vulnerable}")
print(f"🟢 LEAST VULNERABLE: {least_vulnerable}")
print("="*60)

## COP32 Position Paper: Five Key Negotiation-Grade Findings

Based on the comparative climate analysis above, Ethiopia can frame these findings to strengthen Africa's climate diplomacy.

### FINDING 1: FASTEST WARMING COUNTRY

**Identify the country with the steepest warming trend over 2015-2026.**

Add analysis of temperature trend slopes to determine which country is warming fastest.
This strengthens the argument for adaptation finance and loss-and-damage support.

---

### FINDING 2: MOST UNSTABLE PRECIPITATION COUNTRY

**Identify the country with highest rainfall unpredictability (coefficient of variation).**

Countries with CV > 1.0 face extreme rainfall variability → agriculture becomes gambling.
Justifies demand for early warning systems and agricultural insurance schemes.

---

### FINDING 3: COMPOUND CLIMATE STRESS HOTSPOT

**Which country faces simultaneous heat + drought stress?**

Countries with both high heat days AND high dry days face compounding impacts.
This is the strongest argument for adaptation finance and loss-and-damage support.

---

### FINDING 4: ETHIOPIA'S REGIONAL POSITION

**How does Ethiopia compare to its neighbors?**

Use data to show:
- Where Ethiopia ranks in vulnerability
- Whether Ethiopia is a regional leader in climate action readiness
- How Ethiopia can champion regional adaptation finance

---

### FINDING 5: COALITION BUILDING OPPORTUNITY

**Which countries should Ethiopia align with to strengthen bargaining power?**

Countries with similar vulnerability patterns can form a coalition for:
- Joint loss-and-damage claims
- Regional adaptation projects
- Shared water resource management

Ethiopia as COP32 host can lead regional coordination.